# Pipeline Overview

Documents the SAM2-guided segmentation pipeline: yield, mask quality, rejection breakdown,
and the composite scoring formula used to select the best candidate mask per image.


## Setup

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

DATA_DIR = Path('../data')   # override if running from a different working directory
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})


## Mask ranking formula

Each image produces up to 3 candidate masks from the SAM2 predictor. Candidates are first
filtered to those whose major axis falls within the accepted diameter range (30–150 µm), then
ranked by the composite score below. The mask with the highest score is retained.

$$\\text{score} = 0.40 \\times \\text{IoU\_confidence} + 0.35 \\times \\text{solidity} + 0.25 \\times \\text{centrality}$$

$$\\text{centrality} = 1 - \\frac{d(\\text{centroid},\\, \\text{image centre})}{d_{\\max}}$$

- **IoU confidence**: SAM2 predicted intersection-over-union score (0–1).
- **Solidity**: mask area / convex-hull area — penalises merged objects with concave waists.
- **Centrality**: proximity of the mask centroid to the image centre — favours the primary object.

Weights chosen so that segmentation quality (IoU) dominates but shape plausibility (solidity)
and object identity (centrality) break ties.


## Segmentation yield

In [ ]:
morph = pd.read_csv(DATA_DIR / 'hair_morphology_guided.csv', on_bad_lines='skip')

total_input   = 1967   # total TIFFs in Test/input/tiff/
corrupted     = 226    # incomplete transfers — zero pixel data
valid_input   = total_input - corrupted
n_masks       = len(morph)
n_no_mask     = 4      # loaded OK but no candidate passed diameter filter

summary = pd.DataFrame({
    'Metric': [
        'Total images',
        'Excluded (corrupted transfer)',
        'Valid images',
        'Successful masks',
        'No mask detected',
        'Yield (all images)',
        'Yield (valid images only)',
    ],
    'Value': [
        total_input,
        corrupted,
        valid_input,
        n_masks,
        n_no_mask,
        f'{n_masks / total_input * 100:.1f}%',
        f'{n_masks / valid_input * 100:.1f}%',
    ],
})
summary


## IoU confidence distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(morph['confidence'].dropna(), bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(morph['confidence'].median(), color='crimson', linestyle='--', label=f"Median {morph['confidence'].median():.3f}")
ax.set_xlabel('SAM2 predicted IoU score')
ax.set_ylabel('Count')
ax.set_title('Mask confidence distribution (guided pipeline)')
ax.legend()
plt.tight_layout()
plt.show()
print(morph['confidence'].describe().round(3))


## Rejection breakdown

In [ ]:
flag_counts = morph['flag'].fillna('').value_counts()
print('Flag distribution:')
print(flag_counts.to_string())
